In [ ]:
#setup
import io
import math
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

pd.options.display.precision = 3
pd.options.display.max_columns = 30
pd.options.display.width = 140
rng = np.random.default_rng(42)

PROJECT = Path.cwd().parent
DATA = PROJECT / "data"
RAW_FILE = DATA / "sales_messy.csv"
PRODUCT_CATEGORY = {"Notebook": "Stationery", "Pen": "Stationery", "Marker": "Stationery",
                    "Bag": "Accessories", "Bottle": "Accessories",
                    "Headphones": "Electronics", "Mouse": "Electronics"}

def best_time(f, repeat=3):
    best = float("inf")
    for _ in range(repeat):
        start = time.perf_counter()
        f()
        best = min(best, time.perf_counter() - start)
    return best

print(sys.executable)
print("pandas", pd.__version__)
if not RAW_FILE.exists():
    raise FileNotFoundError(f"{RAW_FILE} not found. Run Cell 2 of the S10 notebook first, because it creates the file.")

raw = pd.read_csv(RAW_FILE, parse_dates=["order_date"])         # the ORIGINAL: we never change it
print("raw shape:", raw.shape)

In [ ]:
#Adding and removing columns and rows
#adding columns
t = pd.DataFrame({"item": ["pen", "book", "bag"], "price": [10.0, 50.0, 400.0], "qty": [3, 1, 2]})
t["total"] = t["price"] * t["qty"]                          # a new column from existing ones
t["currency"] = "INR"                                       # a single value fills the whole column
t["big"] = t["total"] > 100                                 # a boolean column
t.insert(1, "sku", ["P1", "B1", "G1"])                      # at a chosen position (column 1)
t2 = t.assign(tax=lambda d: (d["total"] * 0.18).round(2),    # assign returns a NEW frame (good for chains)
              label=lambda d: d["item"].str.upper())
print(t2)
print("t has 'tax':", "tax" in t.columns, "| t2 has 'tax':", "tax" in t2.columns)

In [ ]:
#removing and renaming columns
print(t2.drop(columns=["currency", "big"]).columns.tolist())      # drop returns a NEW frame
popped = t2.pop("label")                                          # removes it from t2 AND returns it
del t2["tax"]                                                     # removes in place
print(popped.tolist(), t2.columns.tolist())

try:
    t2.drop(columns=["nope"])
except KeyError as e:
    print("KeyError:", e)
print(t2.drop(columns=["nope"], errors="ignore").shape, "<- errors='ignore' skips names that are not there")

t3 = t2.rename(columns={"total": "amount"})
print(t3[["item", "qty", "price", "amount"]].columns.tolist(), "<- a list of names reorders the columns")
print(t3.columns.str.upper().tolist())

In [ ]:
#adding and removing rows
people = pd.DataFrame({"name": ["Asha", "Ravi"], "age": [28, 35]})
people.loc[len(people)] = ["Meera", 41]                           # one row at the next label
new_rows = pd.DataFrame({"name": ["Kabir", "Isha"], "age": [22, 30]})
people = pd.concat([people, new_rows], ignore_index=True)         # several rows (DataFrame.append was removed in pandas 2)
print(people, "\n")

print("drop by LABEL    :", people.drop(index=[0, 2])["name"].tolist())
print("keep by CONDITION:", people[people["age"] >= 30]["name"].tolist())
young = people.index[(people["age"] < 25).to_numpy()]
print("drop the matches :", people.drop(young)["name"].tolist())

sub = people[people["age"] > 25].copy()                           # .copy() when you will modify the subset
sub["senior"] = sub["age"] > 35
print(sub)

In [ ]:
#map
s = raw["channel"]
print(s.map({"web": "Online", "app": "Online", "store": "Offline"}).value_counts().to_dict())

partial = s.map({"web": "Online"})                                 # app and store are not in the dict
print("unmapped values became NaN:", int(partial.isna().sum()), "rows")

print(s.map(str.upper).head(3).tolist(), s.map(lambda v: v[0]).head(3).tolist(), s.map("channel: {}".format).head(2).tolist())
print(raw["discount"].map(lambda d: "none" if d == 0 else f"{round(d * 100)}%").value_counts().to_dict())
print(raw["rating"].map(lambda v: f"{int(v)} stars").isna().sum() if False else
      raw["rating"].map(lambda v: "x", na_action="ignore").isna().sum() == raw["rating"].isna().sum())

In [ ]:
#replace and the string tools
print(raw["city"].replace({"delhi": "Delhi"}).value_counts(dropna=False).to_dict())   # exact matches only; the rest stay
print(raw["city"].str.replace(r"\s+$", "", regex=True).isin(["Pune"]).sum(), "<- a regex removes trailing spaces")

x = pd.Series([" Delhi ", "DELHI", "new  delhi", None])
print(x.str.strip().str.title().tolist())
print(x.str.strip().str.lower().str.replace(r"\s+", " ", regex=True).tolist())
print(x.str.contains("del", case=False, na=False).tolist(), x.str.len().tolist())
print(pd.Series(["INR 1,299", "Rs. 45"]).str.extract(r"(\d[\d,]*)")[0].str.replace(",", "").astype(int).tolist())

In [ ]:
#apply
num = raw[["qty", "unit_price", "revenue"]]
print(num.apply(np.max).to_dict(), "<- axis=0 (default): one result per COLUMN")
print(num.apply(lambda c: c.max() - c.min()).round(1).to_dict())
print(num.head(3).apply(lambda r: r["qty"] * r["unit_price"], axis=1).tolist(), "<- axis=1: one result per ROW")

def describe_row(r):
    return pd.Series({"big": r["revenue"] > 3000, "units": int(r["qty"])})

print(raw.head(3).apply(describe_row, axis=1))                    # returning a Series expands into columns

print(raw[["city", "product"]].head(3).map(lambda v: v.upper() if isinstance(v, str) else v))   # DataFrame.map: every cell

In [ ]:
#apply(axis=1) is a loop in disguise
big = pd.concat([raw] * 100, ignore_index=True)                   # 150,600 rows

def band_row(r):
    if r["revenue"] >= 3000:
        return "high"
    if r["revenue"] >= 500:
        return "mid"
    return "low"

by_apply = lambda: big.apply(band_row, axis=1)
by_select = lambda: np.select([big["revenue"] >= 3000, big["revenue"] >= 500], ["high", "mid"], default="low")
by_cut = lambda: pd.cut(big["revenue"], [-np.inf, 500, 3000, np.inf], labels=["low", "mid", "high"], right=False)

t_apply, t_select, t_cut = best_time(by_apply, 1), best_time(by_select), best_time(by_cut)
print(f"apply(axis=1): {t_apply:.2f} s | np.select: {t_select * 1000:.0f} ms | pd.cut: {t_cut * 1000:.0f} ms")
print(f"vectorised is about {t_apply / t_select:.0f}x faster | same answer:",
      bool((by_apply() == by_select()).all()), bool((by_cut().astype(str) == by_select()).all()))

In [ ]:
#Sorting
#sort_values
cols = ["order_id", "city", "product", "qty", "revenue"]
print(raw.sort_values("revenue", ascending=False).head(3)[cols], "\n")
print(raw.sort_values(["city", "revenue"], ascending=[True, False]).head(4)[cols], "\n")      # one direction per key
print(raw.sort_values("city", na_position="first").head(2)[cols], "<- missing values first (the default is last)\n")
print(raw.nlargest(3, "revenue")[["order_id", "revenue"]].index.tolist(), raw.nsmallest(2, "revenue").index.tolist())

by_rank = raw.assign(rev_rank=raw["revenue"].rank(ascending=False, method="min")).nsmallest(3, "rev_rank")
print(by_rank[["order_id", "revenue", "rev_rank"]])

ordered = raw.sort_values("revenue", ascending=False)
print("index after sorting:", ordered.index[:5].tolist(), "-> labels no longer 0, 1, 2 ...")
print("after reset_index(drop=True):", ordered.reset_index(drop=True).index[:5].tolist())
print("sorting does not change raw:", raw.index[:3].tolist())

In [ ]:
#sorting traps
print(pd.Series(["b", "A", "c", "B"]).sort_values().tolist(), "<- capitals first (code-point order)")
print(pd.Series(["b", "A", "c", "B"]).sort_values(key=lambda s: s.str.lower()).tolist(), "<- key= for a case-blind sort")
print(pd.Series(["10", "9", "100"]).sort_values().tolist(), "<- numbers stored as TEXT sort like words")
print(pd.to_numeric(pd.Series(["10", "9", "100"])).sort_values().tolist())

order = pd.CategoricalDtype(["web", "app", "store"], ordered=True)          # your own order
custom = raw.assign(channel=raw["channel"].astype(order)).sort_values("channel", kind="stable")
print(custom["channel"].head(2).tolist(), custom["channel"].tail(2).tolist())
print(raw.set_index("order_id").sort_index(ascending=False).index[:3].tolist())

In [ ]:
#concat
a = pd.DataFrame({"id": [1, 2], "x": ["a", "b"]})
b = pd.DataFrame({"id": [3, 4], "x": ["c", "d"]})
c = pd.DataFrame({"id": [5], "y": [9.5]})
print(pd.concat([a, b]).index.tolist(), "<- the index repeats")
print(pd.concat([a, b], ignore_index=True).index.tolist())
print(pd.concat([a, c]), "\n")                                     # a column missing from one frame becomes NaN
print(pd.concat([a, c], join="inner"), "\n")                       # keep only the shared columns
print(pd.concat([a, b], keys=["first", "second"]), "\n")           # label where each row came from
print(pd.concat([a, c.rename(columns={"id": "id2"})], axis=1))     # side by side, aligned on the index

In [ ]:
#the four merge types
emp = pd.DataFrame({"emp_id": [1, 2, 3, 4], "dept_id": [10, 20, 20, 99], "name": ["Asha", "Ravi", "Meera", "Kabir"]})
dept = pd.DataFrame({"dept_id": [10, 20, 30], "dept": ["IT", "HR", "Ops"]})
for how in ("inner", "left", "right", "outer"):
    print(f"{how:<6}: {len(emp.merge(dept, on='dept_id', how=how))} rows")
print(emp.merge(dept, on="dept_id", how="outer", indicator=True))
print(emp.join(dept.set_index("dept_id"), on="dept_id").equals(emp.merge(dept, on="dept_id", how="left")), "<- join on a key = left merge")

In [ ]:
#keys with different names, clashing columns, and NaN keys
orders = pd.DataFrame({"customer": ["a", "b"], "total": [100, 200]})
prices = pd.DataFrame({"cust_id": ["a", "c"], "total": [999, 5]})
print(orders.merge(prices, left_on="customer", right_on="cust_id", how="outer", suffixes=("_orders", "_prices")))

left = pd.DataFrame({"k": ["a", np.nan], "v": [1, 2]})
right = pd.DataFrame({"k": [np.nan, "b"], "w": [10, 20]})
print(left.merge(right, on="k", how="inner"), "<- the two NaN keys MATCH each other (unlike SQL)")

try:
    pd.DataFrame({"k": [1]}).merge(pd.DataFrame({"k": ["1"]}), on="k")
except ValueError as e:
    print("ValueError:", str(e)[:75])

In [ ]:
#a real lookup join, and what indicator reveals
cities = pd.DataFrame({"city": ["Delhi", "Mumbai", "Pune", "Chennai", "Bengaluru"],
                       "region": ["North", "West", "West", "South", "South"],
                       "manager": ["Rao", "Iyer", "Khan", "Das", "Nair"]})

m = raw.merge(cities, on="city", how="left", indicator=True)
print(len(raw), "->", len(m), "rows (a left join on a UNIQUE key keeps the row count)")
print(m["_merge"].value_counts().to_dict())
print("unmatched city labels:", m.loc[m["_merge"] == "left_only", "city"].value_counts(dropna=False).to_dict())

fixed = raw.assign(city=raw["city"].str.strip().str.title())              # clean the key, then try again
m2 = fixed.merge(cities, on="city", how="left", indicator=True)
print("after cleaning the text:", m2["_merge"].value_counts().to_dict())
print("still unmatched:", m2.loc[m2["_merge"] == "left_only", "city"].value_counts(dropna=False).to_dict())

In [ ]:
#duplicate keys multiply rows
suppliers = pd.DataFrame({"product": ["Pen", "Pen", "Bag", "Mouse"], "supplier": ["Acme", "Zenith", "Bagco", "Techno"]})
exploded = fixed.merge(suppliers, on="product", how="left")
print(f"rows {len(fixed)} -> {len(exploded)} | total revenue {fixed['revenue'].sum():,.0f} -> {exploded['revenue'].sum():,.0f}")
try:
    fixed.merge(suppliers, on="product", how="left", validate="many_to_one")
except pd.errors.MergeError as e:
    print("MergeError:", str(e)[:80])
print("unique lookup passes the check:", len(fixed.merge(cities, on="city", how="left", validate="many_to_one")) == len(fixed))

In [ ]:
#audit, and does missingness depend on anything?
df = raw.copy()
df["customer_age"] = df["customer_age"].where(df["customer_age"].between(18, 75))     # impossible ages become missing
audit = pd.DataFrame({"missing": df.isna().sum(), "percent": (df.isna().mean() * 100).round(1)})
print(audit[audit["missing"] > 0], "\n")
print("rows with at least one missing value:", int(df.isna().any(axis=1).sum()), f"({df.isna().any(axis=1).mean():.0%})")
print("missing values per row:", df.isna().sum(axis=1).value_counts().sort_index().to_dict(), "\n")

for col in ["rating", "customer_age"]:
    by_channel = df.groupby("channel")[col].apply(lambda s: round(s.isna().mean(), 3)).to_dict()
    chi2, p, _, _ = stats.chi2_contingency(pd.crosstab(df["channel"], df[col].isna()))
    print(f"{col:<13} share missing by channel {by_channel} | chi-square p = {p:.3f}")

In [ ]:
#dropping
print("rows:", len(df))
print("dropna()                    :", len(df.dropna()), "<- ANY missing value removes the row")
print("dropna(subset=['city'])     :", len(df.dropna(subset=["city"])))
print("dropna(how='all')           :", len(df.dropna(how="all")))
print("dropna(thresh=12)           :", len(df.dropna(thresh=12)), "<- keep rows with at least 12 non-missing values")
print("dropna(axis=1, thresh=1400) :", df.dropna(axis=1, thresh=1400).shape[1], "of", df.shape[1], "columns kept")

complete = df.dropna()
print(f"\nkept {len(complete) / len(df):.0%} of the rows")
print(f"median revenue: all rows {df['revenue'].median():.1f} | complete rows only {complete['revenue'].median():.1f}")
print(f"return rate   : all rows {df['returned'].mean():.3f} | complete rows only {complete['returned'].mean():.3f}")

In [ ]:
#filling
age = df["customer_age"]
print("constant :", df["city"].fillna("Unknown").value_counts().head(2).to_dict())
print("median   :", age.fillna(age.median()).isna().sum(), "missing left | mean", round(age.mean(), 2), "| median", age.median())
group_median = df.groupby("category")["customer_age"].transform("median")
print("group    :", age.fillna(group_median).isna().sum(), "missing left (a median per category)")

df["rating_missing"] = df["rating"].isna()                         # an indicator keeps the fact that it was missing
print("share who never rated:", round(df["rating_missing"].mean(), 3))

temps = pd.Series([20.0, np.nan, np.nan, 26.0, np.nan, 25.0], index=pd.date_range("2026-10-01", periods=6))
print("ffill      :", temps.ffill().tolist())                      # carry the last value forward
print("bfill      :", temps.bfill().tolist())
print("interpolate:", temps.interpolate().round(2).tolist(), "<- a straight line between known points")

In [ ]:
#which fill method works best?
def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

def imputation_test(frame, column, group, frac=0.2, seed=1):
    """Hide a random share of KNOWN values, fill them in, and measure the error against the truth."""
    known = frame[frame[column].notna()].copy()
    r = np.random.default_rng(seed)
    hide = r.random(len(known)) < frac
    truth = known.loc[hide, column]
    test = known[column].mask(hide)
    group_median = test.groupby(known[group]).transform("median")
    return {"overall mean": rmse(truth, np.full(len(truth), test.mean())),
            "overall median": rmse(truth, np.full(len(truth), test.median())),
            "group median": rmse(truth, group_median[hide])}

results = pd.DataFrame({
    "customer_age, grouped by category": imputation_test(df, "customer_age", "category"),
    "unit_price, grouped by product": imputation_test(df, "unit_price", "product"),
}).round(2)
print(results)
print("\nspread of the columns themselves (SD):", round(df["customer_age"].std(), 2), "and", round(df["unit_price"].std(), 2))

In [ ]:
#mean-filling shrinks the spread
ages = df["customer_age"].dropna().reset_index(drop=True)
hidden = ages.mask(rng.random(len(ages)) < 0.4)                           # hide 40 percent
filled = hidden.fillna(hidden.mean())
print(f"true SD {ages.std():.2f} | SD after mean-filling 40% of the values {filled.std():.2f} "
      f"(about {ages.std() * math.sqrt(0.6):.2f} expected)")
print("the mean barely moves:", round(ages.mean(), 2), "->", round(filled.mean(), 2))

In [ ]:
#exact duplicates in the sales data
dups = raw[raw.duplicated(keep=False)].sort_values("order_id")
print("rows involved:", len(dups), "| exact duplicates:", int(raw.duplicated().sum()), "| repeated order_ids:", int(raw["order_id"].duplicated().sum()))
print(dups[["order_id", "order_date", "city", "product", "qty", "revenue"]].head(4), "\n")

print("keep='first' (default):", len(raw.drop_duplicates()))
print("keep='last'           :", len(raw.drop_duplicates(keep="last")))
print("keep=False (drop ALL copies, even the original):", len(raw.drop_duplicates(keep=False)))
print("subset=['order_id']   :", len(raw.drop_duplicates(subset=["order_id"])), "<- same here, because the copies are exact")
print("a careless subset ['city','product','channel'] would 'find'", int(raw.duplicated(subset=["city", "product", "channel"]).sum()), "duplicates")

In [ ]:
#the same entity, written differently
cust = pd.DataFrame({"email": ["a@x.com", "a@x.com", "b@y.com", "B@Y.COM ", "c@z.com"],
                     "name": ["Asha", "Asha K", "Ravi", "Ravi", "Meera"], "orders": [3, 5, 2, 2, 1]})
print("exact duplicates         :", int(cust.duplicated().sum()))
print("on email as typed        :", int(cust.duplicated(subset=["email"]).sum()))
cust["email_key"] = cust["email"].str.strip().str.lower()                 # a cleaned KEY, kept separate from the data
print("rows in a duplicate group:", int(cust.duplicated(subset=["email_key"], keep=False).sum()))

best = cust.sort_values("orders", ascending=False).drop_duplicates(subset="email_key", keep="first")
print("rule 'keep the row with the most orders':\n", best.sort_index()[["email", "name", "orders"]])
merged = cust.groupby("email_key").agg(name=("name", "first"), orders=("orders", "max"), n_records=("email", "size"))
print("or combine the records:\n", merged)

In [ ]:
#survey
survey_text = """id,age,income,joined,newsletter,city
1,25,"45,000",2025-01-15,yes,Delhi
2,thirty,52000,15/02/2025,Y,delhi
3,41,N/A,2025-03-01,no,Pune 
4,-5,"1,20,000",2025-13-40,TRUE,Mumbai
5,29,61000.50,2025-04-10,n,
6,200,48000,,No,PUNE
"""
sv = pd.read_csv(io.StringIO(survey_text))
orig = sv.copy()
print({k: str(v) for k, v in sv.dtypes.items()})

sv["age_num"] = pd.to_numeric(sv["age"], errors="coerce")                              # 'thirty' becomes NaN
sv["age_clean"] = sv["age_num"].where(sv["age_num"].between(16, 100))                  # -5 and 200 become NaN
sv["income_num"] = pd.to_numeric(sv["income"].str.replace(",", "", regex=False), errors="coerce")
sv["joined_clean"] = pd.to_datetime(sv["joined"], format="%Y-%m-%d", errors="coerce")
yes_no = {"yes": True, "y": True, "true": True, "no": False, "n": False, "false": False}
sv["newsletter_clean"] = sv["newsletter"].str.strip().str.lower().map(yes_no)
sv["city_clean"] = sv["city"].str.strip().str.title()
print(sv[["id", "age_clean", "income_num", "joined_clean", "newsletter_clean", "city_clean"]], "\n")

print("what 'coerce' turned into missing in the age column:\n", sv.loc[sv["age_num"].isna() & orig["age"].notna(), ["id", "age"]])
print("what 'coerce' turned into NaT in the date column:\n", sv.loc[sv["joined_clean"].isna() & orig["joined"].notna(), ["id", "joined"]])
print("unmapped newsletter answers:", int((sv["newsletter_clean"].isna() & orig["newsletter"].notna()).sum()))

In [ ]:
#validation rules for the sales data
RULES = {
    "age outside 18 to 75": lambda d: d["customer_age"].notna() & ~d["customer_age"].between(18, 75),
    "qty above 20": lambda d: d["qty"] > 20,
    "qty not positive": lambda d: d["qty"] <= 0,
    "discount outside 0 to 0.5": lambda d: ~d["discount"].between(0, 0.5),
    "rating outside 1 to 5": lambda d: d["rating"].notna() & ~d["rating"].between(1, 5),
    "date outside 2026": lambda d: ~d["order_date"].between("2026-01-01", "2026-12-31"),
    "category does not match product": lambda d: d["category"] != d["product"].map(PRODUCT_CATEGORY),
    "revenue != qty*price*(1-discount)": lambda d: ~np.isclose(
        d["revenue"], (d["qty"] * d["unit_price"] * (1 - d["discount"])).round(2), atol=0.01),
    "city label has stray spaces or odd case": lambda d: d["city"].notna() & (d["city"] != d["city"].str.strip().str.title()),
}

def validate(frame):
    return pd.Series({name: int(rule(frame).sum()) for name, rule in RULES.items()}, name="violations")

report = validate(raw)
print(report)
print("\nthe offending revenue rows:\n",
      raw.loc[RULES["revenue != qty*price*(1-discount)"](raw), ["order_id", "qty", "unit_price", "discount", "revenue"]])

In [ ]:
#handling extreme values
q99 = df["qty"].quantile(0.99)
variants = {
    "as is": df["qty"],
    "flagged rows removed (qty <= 20)": df.loc[df["qty"] <= 20, "qty"],
    f"capped at the 99th percentile ({q99:.0f})": df["qty"].clip(upper=q99),
    "log1p transform": np.log1p(df["qty"]),
}
print(pd.DataFrame({k: {"mean": v.mean(), "median": v.median(), "SD": v.std(), "max": v.max()} for k, v in variants.items()}).T.round(2))
print("\nthe 3 extreme orders hold", f"{df.loc[df['qty'] > 20, 'revenue'].sum() / df['revenue'].sum():.0%}", "of ALL revenue")

In [ ]:
#A reusable cleaning pipeline
#one function, a log, and final checks
def step_text(df, log):
    out = df.copy()
    changed = 0
    for col in ["city", "product", "channel", "category"]:
        cleaned = out[col].str.strip()
        if col == "city":
            cleaned = cleaned.str.title()
        changed += int((cleaned.fillna("") != out[col].fillna("")).sum())
        out[col] = cleaned
    log.append({"step": "standardise text", "rows": len(out), "changed": changed,
                "note": "strip spaces; title-case city"})
    return out

def step_duplicates(df, log):
    before = len(df)
    out = df.drop_duplicates().reset_index(drop=True)
    log.append({"step": "drop exact duplicates", "rows": len(out), "changed": before - len(out),
                "note": "kept the first copy"})
    return out

def step_errors(df, log):
    out = df.copy()
    bad_age = out["customer_age"].notna() & ~out["customer_age"].between(18, 75)
    out["age_invalid"] = bad_age
    out["customer_age"] = out["customer_age"].mask(bad_age)
    log.append({"step": "impossible ages", "rows": len(out), "changed": int(bad_age.sum()),
                "note": "set to missing; flagged in age_invalid"})
    out["qty_suspect"] = out["qty"] > 20
    log.append({"step": "extreme quantities", "rows": len(out), "changed": int(out["qty_suspect"].sum()),
                "note": "qty > 20 flagged in qty_suspect, NOT deleted"})
    expected = (out["qty"] * out["unit_price"] * (1 - out["discount"])).round(2)
    wrong = ~np.isclose(out["revenue"], expected, atol=0.01)
    out.loc[wrong, "revenue"] = expected[wrong]
    log.append({"step": "recompute revenue", "rows": len(out), "changed": int(wrong.sum()),
                "note": "from qty * unit_price * (1 - discount)"})
    return out

def step_missing(df, log):
    out = df.copy()
    n = int(out["city"].isna().sum())
    out["city"] = out["city"].fillna("Unknown")
    out["has_rating"] = out["rating"].notna()
    log.append({"step": "missing values", "rows": len(out), "changed": n,
                "note": "city -> 'Unknown'; ages and ratings left missing on purpose; has_rating flag added"})
    return out

def step_types(df, log):
    out = df.copy()
    for col in ["city", "product", "channel", "category"]:
        out[col] = out[col].astype("category")
    log.append({"step": "set dtypes", "rows": len(out), "changed": 4, "note": "text columns -> category"})
    return out

def clean_sales(raw):
    log = []
    clean = (raw.pipe(step_text, log).pipe(step_duplicates, log).pipe(step_errors, log)
                .pipe(step_missing, log).pipe(step_types, log))
    assert clean["order_id"].is_unique, "order_id is not unique"                    # fail LOUDLY if a rule still breaks
    assert (clean["qty"] > 0).all(), "non-positive quantity"
    assert clean["customer_age"].dropna().between(18, 75).all(), "an impossible age is left"
    assert clean["city"].notna().all(), "a missing city is left"
    return clean, pd.DataFrame(log)

clean, log_df = clean_sales(raw)
print(log_df.to_string(index=False))
print("\nraw", raw.shape, "-> clean", clean.shape)

In [ ]:
#what did cleaning change?
def summary(frame):
    return {"rows": len(frame), "duplicate rows": int(frame.duplicated().sum()),
            "distinct city labels (incl. missing)": frame["city"].nunique(dropna=False),
            "missing city": int(frame["city"].isna().sum()),
            "impossible ages": int((frame["customer_age"].notna() & ~frame["customer_age"].between(18, 75)).sum()),
            "total revenue": round(float(frame["revenue"].sum()), 2),
            "mean qty": round(float(frame["qty"].mean()), 3),
            "mean age": round(float(frame["customer_age"].mean()), 3)}

print(pd.DataFrame({"raw": summary(raw), "clean": summary(clean)}))
print("\nrules violated after cleaning:\n", validate(clean)[lambda s: s > 0])

In [ ]:
#save the results for the next days
clean.to_csv(DATA / "sales_clean.csv", index=False)
log_df.to_csv(DATA / "cleaning_log.csv", index=False)
check = pd.read_csv(DATA / "sales_clean.csv", parse_dates=["order_date"])
print(check.shape, "| dtypes survived the round trip?", str(check["city"].dtype), "(category info is lost in a CSV, so S12 reloads and converts again)")
print(sorted(p.name for p in DATA.iterdir()))